# Extracting a Precipitation Time Series from RADOLAN Data

This notebook walks through the full workflow for turning DWD RADOLAN radar data into a
precipitation time series for a **single geographic location**.

RADOLAN data covers Germany on a fixed **900 × 900** grid. Each grid cell is roughly
1 km × 1 km and holds an hourly precipitation value. To get a time series for a point of
interest (e.g. a gauge, a city, or a catchment outlet) we:

1. **Configure** the date range and target coordinate.
2. **Download** the raw RADOLAN archives from the DWD open-data server (one-time step).
3. **Locate** the grid pixel closest to our coordinate.
4. **Build a boolean mask** selecting the pixel (and a small neighbourhood around it).
5. **Extract and aggregate** the masked pixels into a single time series and save it.

**Prerequisites:** install the package (`pip install dwd-radolan-utils` or `uv sync --dev`
from a source checkout). Downloading several years of data can take a while and needs a few
GB of disk space, so the download step is gated behind a flag and only needs to run once.

## Imports

We use the three public entry points of `dwd_radolan_utils` — `download_dwd` to fetch the raw
data, `get_wgs84_grid` to map coordinates onto the radar grid, and
`extract_time_series_from_radar` to build the series — plus `numpy` for the masking maths.

In [20]:
from datetime import datetime
from pathlib import Path

import numpy as np

from dwd_radolan_utils.download import download_dwd
from dwd_radolan_utils.extraction import extract_time_series_from_radar
from dwd_radolan_utils.geo_utils import get_wgs84_grid

## 1. Configuration

Set the parameters for the run in one place:

- **`DOWNLOAD_NEW_DATA`** — set to `True` the first time to fetch the raw data, then back to
  `False` on subsequent runs so you don't re-download.
- **`PRE_NAME`** — the column name used for the extracted precipitation series in the output.
- **`start_date` / `end_date`** — the time window to extract.
- **`radar_data_dir`** — where the downloaded/processed RADOLAN `.npz` files live.
- **`target_lat` / `target_lon`** — the WGS84 coordinate (latitude, longitude) you want a
  time series for. The default points at Stadtlohn, Germany.

In [21]:
DOWNLOAD_NEW_DATA = False
PRE_NAME = "col_name"

start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 12, 31)

OUTPUT_NAME = f"ts_{start_date.strftime('%Y-%m-%d')}_{end_date.strftime('%Y-%m-%d')}"

radar_data_dir = Path("../data/dwd/processed_data")
radar_data_dir.mkdir(parents=True, exist_ok=True)

target_lat = 51.994682
target_lon = 6.915830

## 2. Download the historical RADOLAN data

`download_dwd` fetches the raw RADOLAN archives from the DWD open-data server, decodes them,
and stores them as compressed `.npz` files in `save_path`. The `type_radolan` argument
selects the data source:

- `"historical"` — archived data for past years (used here).
- `"recent"` — data for the current year.
- `"now"` — the current day.

> **Note:** You only need to run this once per date range. Downloading several years of
> historical data can take a while (often tens of minutes) and uses a few GB of disk. Keep
> `DOWNLOAD_NEW_DATA = False` afterwards to skip straight to extraction.

In [22]:
if DOWNLOAD_NEW_DATA:
    download_dwd(type_radolan="historical", start=start_date, end=end_date, save_path=radar_data_dir)

## 3. Find the RADOLAN grid pixel closest to your coordinate

The RADOLAN grid uses its own polar-stereographic projection, so we can't index it directly
with a latitude/longitude. `get_wgs84_grid()` returns the WGS84 coordinate of every grid cell
as an array of shape `(900, 900, 2)`, where index `0` is latitude and `1` is longitude.

We compute the squared distance from our target coordinate to every cell and take the
`argmin` to find the `(row, col)` index of the nearest pixel. (Squared Euclidean distance in
lat/lon is good enough at this scale to pick the closest cell, and avoids an unnecessary
square root.)

In [23]:
wgs84_grid = get_wgs84_grid()  # Output shape: (900, 900, 2), where index 0 is latitude and 1 is longitude
distances = (wgs84_grid[:, :, 0] - target_lat) ** 2 + (wgs84_grid[:, :, 1] - target_lon) ** 2
closest_idx = np.unravel_index(np.argmin(distances), distances.shape)

print(f"Closest RADOLAN grid index to ({target_lat}, {target_lon}) is: {closest_idx}")

Closest RADOLAN grid index to (51.994682, 6.91583) is: (np.int64(565), np.int64(287))


## 4. Build the boolean grid mask

`extract_time_series_from_radar` expects a **3D boolean mask** of shape
`(number_of_locations, 900, 900)`. Each `(900, 900)` slice marks the cells belonging to one
output location with `True`; every separate slice becomes its own time series. Here we have a
single location, so the shape is `(1, 900, 900)`.

Instead of selecting just the single closest pixel, we select a **3 × 3 block** centred on it.
Averaging over a small neighbourhood smooths out single-cell radar noise and better
represents precipitation around the point. The `max(0, ...)` / `min(900, ...)` guards keep the
block inside the grid bounds when the target sits near an edge.

In [24]:
grid = np.zeros((1, 900, 900), dtype=bool)

# Define a 3x3 grid around the closest coordinate
row_idx, col_idx = closest_idx
row_start, row_end = max(0, row_idx - 1), min(900, row_idx + 2)
col_start, col_end = max(0, col_idx - 1), min(900, col_idx + 2)

# Set the boolean mask to True at our identified 3x3 pixel area
grid[0, row_start:row_end, col_start:col_end] = True
print(f"Selected 3x3 area: rows {row_start}:{row_end}, columns {col_start}:{col_end} {np.sum(grid)} (should be 9 for a full 3x3 area)")

Selected 3x3 area: rows 564:567, columns 286:289 9 (should be 9 for a full 3x3 area)


## 5. Extract the time series

`extract_time_series_from_radar` reads the `.npz` files in `path` that fall within the date
range, crops each radar frame to the bounding box of the mask, and aggregates the selected
cells into one value per time step.

Key arguments:

- **`grid_aggregation_method="mean"`** — averages the 9 cells of our 3×3 block per time step
  (use e.g. `"sum"` or `"max"` for different aggregations).
- **`save=True`** with **`save_path`** and **`save_column_names`** — writes the result to disk
  using `PRE_NAME` as the column header.

It returns `ts_array` (the precipitation values) and `timestamps` (the matching times). The
progress bar tracks the per-file reading of the radar archives.

In [ ]:
ts_array, timestamps = extract_time_series_from_radar(
    grid=grid,
    path=radar_data_dir,
    start_date=start_date,
    end_date=end_date,
    grid_aggregation_method="mean",  # takes the mean over the 3x3 grid
    save=True,
    save_path=Path(OUTPUT_NAME),
    save_column_names=[PRE_NAME],
)

print(f"Extracted {len(timestamps)} time steps into {OUTPUT_NAME}")

(1, 900, 900)


 82%|████████▏ | 9/11 [06:02<01:08, 34.45s/it]

## Done

You now have a precipitation time series saved to disk and available in memory as `ts_array`
(values) and `timestamps` (times).

**Where to go from here:**

- Load the saved file with `pandas` (e.g. `pd.read_csv(...)`) to plot or analyse the series.
- Extract **multiple locations at once** by stacking several masks into the
  `(number_of_locations, 900, 900)` grid — each slice yields its own column.
- Try a different `grid_aggregation_method` (`"sum"`, `"max"`, …) or a larger neighbourhood
  to suit your use case.
- For catchment-based extraction (instead of a fixed pixel block), see
  `compute_catchement_for_location` in `dwd_radolan_utils.catchment_area`.